# Data Exploration Notebook

This notebook helps you explore and understand your parcel data before creating the ETL pipeline.

## Steps:
1. Load the shapefile
2. Inspect schema (columns, data types)
3. View sample data
4. Check data quality
5. Understand geometry structure
6. Generate mapping config template


In [2]:
pip install geopandas pandas numpy shapely psycopg2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
## 1. Setup and Imports

import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import MultiPolygon, Polygon
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful")


✅ Imports successful


In [4]:
## 2. Load Data
#Update the path below to your shapefile location:

SHAPEFILE_PATH = "../data/vellore/vellore_cad.shp"

print(f"📂 Loading shapefile from: {SHAPEFILE_PATH}")
gdf = gpd.read_file(SHAPEFILE_PATH)
print(f"✅ Loaded {len(gdf)} rows")
print(f"📊 Coordinate Reference System: {gdf.crs}")


📂 Loading shapefile from: ../data/vellore/vellore_cad.shp
✅ Loaded 160631 rows
📊 Coordinate Reference System: EPSG:4326


## 3. Inspect Schema

View all columns and their data types:


In [5]:
print("📋 Column Information:")
print("=" * 60)
print(f"Total columns: {len(gdf.columns)}")
print(f"Geometry column: {gdf.geometry.name}")
print("\nColumn Details:")
print("-" * 60)

for col in gdf.columns:
    dtype = gdf[col].dtype
    non_null = gdf[col].notna().sum()
    null_count = gdf[col].isna().sum()
    print(f"{col:30s} | {str(dtype):15s} | Non-null: {non_null:6d} | Null: {null_count:6d}")


📋 Column Information:
Total columns: 13
Geometry column: geometry

Column Details:
------------------------------------------------------------
FID_vellor                     | int64           | Non-null: 160631 | Null:      0
kide                           | object          | Non-null: 160444 | Null:    187
kide_1                         | object          | Non-null:      0 | Null: 160631
kide_2                         | object          | Non-null:      0 | Null: 160631
survey_n_4                     | object          | Non-null: 160004 | Null:    627
gp_name                        | object          | Non-null: 160631 | Null:      0
v_name                         | object          | Non-null: 160631 | Null:      0
v_code                         | object          | Non-null: 160631 | Null:      0
FID_Tamiln                     | int64           | Non-null: 160631 | Null:      0
District                       | object          | Non-null: 160631 | Null:      0
STATE                     

## 4. Sample Data

View first 5 rows to understand the data structure:


In [6]:
print("📄 First 5 Rows (Sample Data):")
print("=" * 60)
display(gdf.head())


📄 First 5 Rows (Sample Data):


,FID_vellor,kide,kide_1,kide_2,survey_n_4,gp_name,v_name,v_code,FID_Tamiln,District,STATE,TEHSIL,geometry
0,0,1,None,None,1,Vediyangadu,VENUGOPALAPURAM,628967,87,VELLORE,TAMIL NADU,KATPADI,"POLYGON ((79.28457 13.12143, 79.2839 13.1214, ..."
1,1,2,None,None,2,Vediyangadu,VENUGOPALAPURAM,628967,87,VELLORE,TAMIL NADU,KATPADI,"POLYGON ((79.28543 13.1202, 79.28522 13.11968,..."
2,2,2,None,None,2,Keeraisathu,KIRAISATHU,630375,87,VELLORE,TAMIL NADU,KATPADI,"POLYGON ((79.28485 13.11978, 79.28469 13.11979..."
3,3,3,None,None,3,Vediyangadu,VENUGOPALAPURAM,628967,87,VELLORE,TAMIL NADU,KATPADI,"POLYGON ((79.28676 13.11733, 79.28658 13.11727..."
4,4,3,None,None,3,Keeraisathu,KIRAISATHU,630375,87,VELLORE,TAMIL NADU,KATPADI,"POLYGON ((79.28658 13.11727, 79.2863 13.11717,..."


## 5. Unique Values Analysis

Check unique values in key columns (helps understand data distribution):


In [7]:
print("🔍 Unique Values Analysis:")
print("=" * 60)

# Check unique values for text columns (first 20)
text_cols = gdf.select_dtypes(include=['object']).columns

for col in text_cols[:10]:  # Limit to first 10 columns
    unique_count = gdf[col].nunique()
    print(f"\n{col}:")
    print(f"  Unique values: {unique_count}")
    if unique_count <= 200:
        print(f"  Values: {gdf[col].unique().tolist()}")
    else:
        print(f"  Sample values: {gdf[col].unique()[:10].tolist()}...")


🔍 Unique Values Analysis:

kide:
  Unique values: 16069
  Sample values: ['1', '2', '3', '33', '34', '35', '4', '53', '54', '55']...

kide_1:
  Unique values: 0
  Values: [None]

kide_2:
  Unique values: 0
  Values: [None]

survey_n_4:
  Unique values: 4706
  Sample values: ['1', '2', '3', '33', '34', '35', '4', '53', '54', '55']...

gp_name:
  Unique values: 246
  Sample values: ['Vediyangadu', 'Keeraisathu', 'uncovered', 'Adukkamparai', 'Thirumani', 'Palamathi', 'Kaniyambadi', 'Palamadai_rf2', 'Vallam', 'Kammavanpettai']...

v_name:
  Unique values: 374
  Sample values: ['VENUGOPALAPURAM', 'KIRAISATHU', 'GUDIYATTAM(M)', 'SATHUVACHARI(M)', 'PARADARAMI R.F', 'ADUKKAMPARAI', 'PERNAMBUT(M)', 'ARIYUR(CT)', 'OLAIYATHUR', 'PALAMADAI R.F']...

v_code:
  Unique values: 390
  Sample values: ['628967', '630375', '803375', '803396', '630351', '630829', '803376', '630913', '630413', '630887']...

District:
  Unique values: 1
  Values: ['VELLORE']

STATE:
  Unique values: 1
  Values: ['TAMIL NADU'

## 6. Geometry Analysis

Understand the geometry structure:


In [11]:
print("🗺️  Geometry Analysis:")
print("=" * 60)

# Geometry types
geom_types = gdf.geometry.geom_type.value_counts()
print(f"\nGeometry Types:")
for geom_type, count in geom_types.items():
    print(f"  {geom_type}: {count}")

# Check for invalid geometries
invalid_count = (~gdf.geometry.is_valid).sum()
print(f"\nInvalid geometries: {invalid_count}")

# Bounding box
bounds = gdf.total_bounds
print(f"\nBounding Box:")
print(f"  Min X (Longitude): {bounds[0]:.6f}")
print(f"  Min Y (Latitude): {bounds[1]:.6f}")
print(f"  Max X (Longitude): {bounds[2]:.6f}")
print(f"  Max Y (Latitude): {bounds[3]:.6f}")


🗺️  Geometry Analysis:

Geometry Types:
  Polygon: 156563
  MultiPolygon: 4068

Invalid geometries: 10

Bounding Box:
  Min X (Longitude): 78.509272
  Min Y (Latitude): 12.658767
  Max X (Longitude): 79.294544
  Max Y (Latitude): 13.171955


In [12]:
# Check geometry validity in detail
if invalid_count > 0:
    print("\n⚠️  Invalid Geometries Found:")
    invalid_gdf = gdf[~gdf.geometry.is_valid]
    print(f"Total invalid: {len(invalid_gdf)}")
    
    # Show first few invalid geometries
    if len(invalid_gdf) > 0:
        print("\nFirst invalid geometry details:")
        display(invalid_gdf.head(3))
else:
    print("\n✅ All geometries are valid!")



⚠️  Invalid Geometries Found:
Total invalid: 10

First invalid geometry details:


,FID_vellor,kide,kide_1,kide_2,survey_n_4,gp_name,v_name,v_code,FID_Tamiln,District,STATE,TEHSIL,geometry
7158,7127,226,None,None,226,uncovered,PALAVANSATHU (CT),630912,292,VELLORE,TAMIL NADU,VELLORE,"POLYGON ((79.13829 12.87711, 79.13817 12.8767,..."
11509,11377,296,None,None,296,Palamathi,VIRUPAKSHIPURAM(CT),630911,292,VELLORE,TAMIL NADU,VELLORE,"POLYGON ((79.14534 12.88465, 79.14566 12.88458..."
15116,15074,267,None,None,267,Melmonavur,MELMANAVUR,630799,292,VELLORE,TAMIL NADU,VELLORE,"POLYGON ((79.09279 12.91649, 79.09268 12.91647..."


## 7. Data Quality Checks

Check for missing values and data quality issues:


In [13]:
print("🔍 Data Quality Report:")
print("=" * 60)

# Missing values
missing = gdf.isnull().sum()
missing_pct = (missing / len(gdf)) * 100

print("\nMissing Values:")
for col in gdf.columns:
    if missing[col] > 0:
        print(f"  {col:30s}: {missing[col]:6d} ({missing_pct[col]:5.2f}%)")

if missing.sum() == 0:
    print("  ✅ No missing values found!")


🔍 Data Quality Report:

Missing Values:
  kide                          :    187 ( 0.12%)
  kide_1                        : 160631 (100.00%)
  kide_2                        : 160631 (100.00%)
  survey_n_4                    :    627 ( 0.39%)


## 8. Identify Key Fields

Based on the exploration, identify which fields map to our target schema:

**Target Schema (parcels_master):**
- `state` (TEXT)
- `state_code` (TEXT)
- `district_code` (TEXT)
- `district_name` (TEXT)
- `sub_district_name` (TEXT) - optional
- `village_name` (TEXT) - optional
- `survey_num` (TEXT)
- `geom` (MultiPolygon, 4326)
- `label_point` (Point, 4326) - computed

**Target Schema (parcels_simplified):**
- `parcel_uuid` (UUID) - from master
- `geom` (MultiPolygon, 4326) - simplified
- `label_point` (Point, 4326) - computed
- `state_code` (TEXT)
- `district_code` (TEXT)
- `survey_num` (TEXT)


## 9. Generate Config Template

Create a YAML config template based on the data structure:


In [17]:
try:
    import yaml
    yaml_available = True
except ImportError:
    print("⚠️  PyYAML not installed. Install with: pip install pyyaml")
    yaml_available = False

if yaml_available:
    # Generate config template with correct field mappings
    config_template = {
        'dataset': {
            'name': 'vellore_cad',
            'source_path': SHAPEFILE_PATH,
            'crs': str(gdf.crs) if gdf.crs else 'EPSG:4326'
        },
        'field_mapping': {
            # Source column -> Target column (direct mappings)
            'STATE': 'state',
            'District': 'district_name',
            'TEHSIL': 'sub_district_name',
            'v_name': 'village_name',
            'survey_n_4': 'survey_num'
        },
        'transforms': {
            # Fields that need transformation
            'state_code': {
                'source': 'FID_Tamiln',  # Source field for state_code
                'transform': 'string'  # Convert int64 to string
            },
            'district_code': {
                'source': 'FID_vellor',  # Source field for district_code
                'transform': 'string'  # Convert int64 to string
            }
        },
        'required_fields': [
            'survey_num',
            'state',
            'state_code',
            'district_code',
            'district_name'
        ],
        'geometry': {
            'simplify_tolerance': 0.00008,
            'target_crs': 'EPSG:4326',
            'compute_label_point': True  # Compute ST_PointOnSurface during ETL
        },
        'notes': {
            'parcel_uuid': 'Auto-generated by database (DEFAULT gen_random_uuid()) - no mapping needed',
            'geom': 'Taken from shapefile geometry column - cleaned and normalized to MultiPolygon',
            'label_point': 'Computed during ETL using ST_PointOnSurface(geom) - moved from tile generation'
        }
    }
    
    print("📝 Generated Config Template:")
    print("=" * 60)
    yaml_content = yaml.dump(config_template, default_flow_style=False, sort_keys=False)
    print(yaml_content)
    
    # Save to config file
    import os
    config_dir = "config"
    config_file = os.path.join(config_dir, "vellore_mapping.yaml")
    
    # Create config directory if it doesn't exist
    os.makedirs(config_dir, exist_ok=True)
    
    # Write YAML to file
    with open(config_file, 'w') as f:
        f.write(yaml_content)
    
    print(f"\n💾 Config saved to: {config_file}")
    print("\n✅ Key Points:")
    print("  • parcel_uuid: Auto-generated by DB (no mapping needed)")
    print("  • geom: Taken from shapefile geometry column (cleaned & normalized)")
    print("  • label_point: Computed during ETL (ST_PointOnSurface) - moved from tile generation")


📝 Generated Config Template:
dataset:
  name: vellore_cad
  source_path: ../data/vellore/vellore_cad.shp
  crs: EPSG:4326
field_mapping:
  STATE: state
  District: district_name
  TEHSIL: sub_district_name
  v_name: village_name
  survey_n_4: survey_num
transforms:
  state_code:
    source: FID_Tamiln
    transform: string
  district_code:
    source: FID_vellor
    transform: string
required_fields:
- survey_num
- state
- state_code
- district_code
- district_name
geometry:
  simplify_tolerance: 8.0e-05
  target_crs: EPSG:4326
  compute_label_point: true
notes:
  parcel_uuid: Auto-generated by database (DEFAULT gen_random_uuid()) - no mapping
    needed
  geom: Taken from shapefile geometry column - cleaned and normalized to MultiPolygon
  label_point: Computed during ETL using ST_PointOnSurface(geom) - moved from tile
    generation


💾 Config saved to: config\vellore_mapping.yaml

✅ Key Points:
  • parcel_uuid: Auto-generated by DB (no mapping needed)
  • geom: Taken from shapefile 